# Análise consolidada de eventos LHE com pandas.DataFrame

Notebook público de roteiro para estudantes do repositório-exemplo da disciplina FIS01214. Ele demonstra uma análise completa do canal de Higgs em dois fótons. A análise percorre as Partes 1 e 2 usando `pandas.DataFrame` como estrutura principal, com células executadas, tabelas, gráficos e resultados numéricos preservados quando este notebook for executado no CERN SWAN. Os outputs devem ser preservados na cópia publicada do exemplo.

## Ambientes suportados

- CERN SWAN: clone o repositório pela interface usando sua URL HTTPS, instale as dependências se necessário e execute o notebook a partir do clone.

Dependências mínimas: `numpy`, `pandas`, `matplotlib` e `IPython`. Este notebook não depende de `pylhe`; a leitura é feita diretamente com Python e convertida para DataFrames.


## Processo demonstrado

Este exemplo usa as amostras de Higgs em dois fótons: sinal `pp → H → γγ` e fundo `pp → γγ` sem contribuição do Higgs. O objetivo é mostrar uma análise completa em nível de gerador, incluindo seleção, distribuições cinemáticas, massa invariante, normalização e interpretação das limitações detectoras.


In [ ]:
# CERN SWAN: verificar e instalar dependências no kernel atual
import importlib.util
import subprocess, sys
pacotes = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "IPython": "ipython"}
ausentes = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if ausentes:
    print("Instalando no kernel SWAN:", ausentes)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *ausentes])
else:
    print("Todas as dependências já estão disponíveis.")
for modulo in pacotes:
    modulo_import = "IPython" if modulo == "IPython" else modulo
    print(modulo, importlib.import_module(modulo_import).__version__ if modulo_import != "IPython" else "disponível")


# Projeto Final - Analise de eventos LHE - referencia comentada

## Parte 1: Leitura do LHE para `pandas.DataFrame`

Este notebook contem uma solucao de referencia para a abordagem **pandas.DataFrame**.
Ele inclui codigo completo e respostas de conferencia para a Parte 1.


## 0. Preparação no CERN SWAN

Na interface do CERN SWAN, clone este repositório usando a URL HTTPS `https://github.com/FIS01214/analise-swan.git`. Abra o notebook a partir da pasta clonada.

Depois, peça ao agente os códigos de cada etapa indicada neste notebook e copie-os nas células de código correspondentes. Os arquivos `data/sinal.lhe.gz` e `data/fundo.lhe.gz` já estão no clone; eles podem ser lidos diretamente com `gzip.open`, sem baixar ZIP, fazer upload ou descompactar o repositório.

Execute primeiro a célula de instalação/verificação de dependências e mantenha os caminhos relativos aos arquivos de dados.


## Organização e roteiro

O notebook segue os quatro itens de `parte1.txt`:

1. leitura dos arquivos LHE;
2. contagem dos eventos e tabela de partículas por `status`;
3. histogramas de $p_T$, $\eta$ e $\phi$ para as partículas finais visíveis;
4. estudo de cortes cinemáticos e reconstrução dos histogramas após a seleção.

> Execute as células na ordem. O notebook localiza automaticamente a raiz do projeto quando é aberto a partir da pasta `notebooks/` ou da raiz do workdir.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import gzip
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 100)

def localizar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for candidato in candidatos:
        if (candidato / "data" / "sinal.lhe.gz").exists():
            return candidato.resolve()
    raise FileNotFoundError(
        "Não encontrei data/sinal.lhe.gz. Abra o notebook a partir da raiz "
        "do projeto ou da pasta notebooks/."
    )

RAIZ = localizar_raiz()
PASTA_DADOS = RAIZ / "data"
PASTA_GRAFICOS = RAIZ / "resultados" / "graficos"
PASTA_GRAFICOS.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "Sinal": PASTA_DADOS / "sinal.lhe.gz",
    "Fundo": PASTA_DADOS / "fundo.lhe.gz",
}

print(f"Raiz do projeto: {RAIZ}")
for amostra, caminho in ARQUIVOS.items():
    tamanho_mb = caminho.stat().st_size / 1024**2
    print(f"{amostra:5s}: {caminho.name} ({tamanho_mb:.2f} MB)")

## 1. Leitura dos arquivos LHE

Um bloco `<event>` começa com uma linha de cabeçalho. O primeiro número dessa linha informa quantas partículas pertencem ao evento. Cada linha seguinte contém, entre outras informações, o identificador PDG, o `status`, as componentes do momento $(p_x,p_y,p_z)$, a energia e a massa.

Usaremos `gzip.open` para ler os arquivos compactados sem criar cópias descompactadas.

In [ ]:
def abrir_lhe(caminho):
    caminho = Path(caminho)
    if caminho.suffix == ".gz":
        return gzip.open(caminho, "rt", encoding="utf-8", errors="replace")
    return caminho.open("rt", encoding="utf-8", errors="replace")

def extrair_processo_madgraph(caminho):
    trecho = []
    dentro_do_cartao = False
    with abrir_lhe(caminho) as arquivo:
        for linha in arquivo:
            if "<MG5ProcCard>" in linha:
                dentro_do_cartao = True
            elif "</MG5ProcCard>" in linha:
                break
            elif dentro_do_cartao:
                trecho.append(linha)
    comandos = re.findall(r"^\s*generate\s+(.+?)\s*$", "".join(trecho), re.MULTILINE)
    return comandos[-1] if comandos else "processo não identificado"

for amostra, caminho in ARQUIVOS.items():
    print(f"{amostra}: {extrair_processo_madgraph(caminho)}")


In [ ]:
def ler_lhe_para_dataframes(arquivos):
    eventos = []
    particulas = []

    for amostra, caminho in arquivos.items():
        dentro_evento = False
        esperando_cabecalho = False
        restantes = 0
        indice_evento = 0

        with abrir_lhe(caminho) as arquivo:
            for numero_linha, linha in enumerate(arquivo, start=1):
                linha = linha.strip()
                if linha == "<event>":
                    dentro_evento = True
                    esperando_cabecalho = True
                    restantes = 0
                    indice_evento += 1
                    continue
                if not dentro_evento:
                    continue
                if esperando_cabecalho:
                    if not linha or linha.startswith("#"):
                        continue
                    campos = linha.split()
                    if len(campos) < 6:
                        raise ValueError(f"Cabeçalho inválido na linha {numero_linha}")
                    nup = int(campos[0])
                    eventos.append({
                        "Amostra": amostra,
                        "Evento": indice_evento,
                        "NUP": nup,
                        "IDPRUP": int(campos[1]),
                        "peso_evento": float(campos[2]),
                        "SCALUP": float(campos[3]),
                        "AQEDUP": float(campos[4]),
                        "AQCDUP": float(campos[5]),
                    })
                    restantes = nup
                    esperando_cabecalho = False
                    continue
                if restantes > 0:
                    campos = linha.split()
                    if len(campos) < 13:
                        raise ValueError(f"Partícula inválida na linha {numero_linha}")
                    particulas.append({
                        "Amostra": amostra,
                        "Evento": indice_evento,
                        "PDG ID": int(campos[0]),
                        "status": int(campos[1]),
                        "mae_1": int(campos[2]),
                        "mae_2": int(campos[3]),
                        "cor_1": int(campos[4]),
                        "cor_2": int(campos[5]),
                        "px": float(campos[6]),
                        "py": float(campos[7]),
                        "pz": float(campos[8]),
                        "energia": float(campos[9]),
                        "massa": float(campos[10]),
                        "tempo_vida": float(campos[11]),
                        "spin": float(campos[12]),
                    })
                    restantes -= 1
                    if restantes == 0:
                        dentro_evento = False

    eventos_df = pd.DataFrame(eventos)
    particulas_df = pd.DataFrame(particulas)
    particulas_df["pT [GeV]"] = np.hypot(particulas_df["px"], particulas_df["py"])
    particulas_df["eta"] = np.arcsinh(particulas_df["pz"] / particulas_df["pT [GeV]"].replace(0, np.nan))
    particulas_df["phi"] = np.arctan2(particulas_df["py"], particulas_df["px"])
    return eventos_df, particulas_df

eventos_df, particulas_df = ler_lhe_para_dataframes(ARQUIVOS)
display(eventos_df.head())
display(particulas_df.head())

@dataclass(frozen=True)
class Particula:
    pdg_id: int
    status: int
    mae_1: int
    mae_2: int
    cor_1: int
    cor_2: int
    px: float
    py: float
    pz: float
    energia: float
    massa: float
    tempo_vida: float
    spin: float

    @property
    def pt(self):
        return math.hypot(self.px, self.py)

    @property
    def eta(self):
        if self.pt == 0:
            return math.copysign(math.inf, self.pz)
        return math.asinh(self.pz / self.pt)

    @property
    def phi(self):
        return math.atan2(self.py, self.px)

@dataclass(frozen=True)
class Evento:
    processo_id: int
    peso: float
    escala: float
    alpha_qed: float
    alpha_qcd: float
    particulas: tuple

def converter_dataframes_para_eventos(eventos_df, particulas_df):
    eventos_por_amostra = {}
    for amostra, tabela_eventos in eventos_df.groupby("Amostra", sort=False):
        lista = []
        tabela_particulas_amostra = particulas_df[particulas_df["Amostra"] == amostra]
        for linha_evento in tabela_eventos.itertuples(index=False):
            partes = tabela_particulas_amostra[tabela_particulas_amostra["Evento"] == linha_evento.Evento]
            particulas = tuple(
                Particula(
                    pdg_id=int(linha["PDG ID"]),
                    status=int(linha["status"]),
                    mae_1=int(linha["mae_1"]),
                    mae_2=int(linha["mae_2"]),
                    cor_1=int(linha["cor_1"]),
                    cor_2=int(linha["cor_2"]),
                    px=float(linha["px"]),
                    py=float(linha["py"]),
                    pz=float(linha["pz"]),
                    energia=float(linha["energia"]),
                    massa=float(linha["massa"]),
                    tempo_vida=float(linha["tempo_vida"]),
                    spin=float(linha["spin"]),
                )
                for _, linha in partes.iterrows()
            )
            dados_evento = linha_evento._asdict()
            lista.append(Evento(
                processo_id=int(dados_evento["IDPRUP"]),
                peso=float(dados_evento["peso_evento"]),
                escala=float(dados_evento["SCALUP"]),
                alpha_qed=float(dados_evento["AQEDUP"]),
                alpha_qcd=float(dados_evento["AQCDUP"]),
                particulas=particulas,
            ))
        eventos_por_amostra[amostra] = lista
    return eventos_por_amostra

def ler_lhe(caminho):
    raise RuntimeError("Nesta versão, use `eventos_df` e `particulas_df`; `eventos` já foi construído a partir dos DataFrames.")

eventos = converter_dataframes_para_eventos(eventos_df, particulas_df)


In [ ]:
for amostra, lista_eventos in eventos.items():
    print(f"Número total de eventos na amostra de {amostra.lower()}: {len(lista_eventos):,}")

print("\nEventos em DataFrame:")
display(eventos_df.groupby("Amostra").size().rename("Eventos").reset_index())

print("\nPartículas em DataFrame:")
display(particulas_df.groupby("Amostra").size().rename("Partículas").reset_index())


## 2a. Investigação preliminar do arquivo

A célula anterior mostra o número de blocos `<event>` efetivamente lidos em cada amostra. Essa contagem deve coincidir com o número solicitado na geração do MadGraph.

In [ ]:
resumo_amostras = pd.DataFrame([
    {
        "Amostra": amostra,
        "Processo no cartão": extrair_processo_madgraph(ARQUIVOS[amostra]),
        "Eventos lidos": len(lista_eventos),
        "Partículas registradas": sum(len(evento.particulas) for evento in lista_eventos),
    }
    for amostra, lista_eventos in eventos.items()
])
display(resumo_amostras)

### Validação da integridade das amostras

Antes de interpretar distribuições físicas, verificamos o número de partículas por evento, os canais partônicos iniciais e a frequência da topologia final esperada. Para este exemplo, a topologia esperada é um par de fótons no estado final. Eventos sem dois fótons devem ser investigados antes da seleção.

In [ ]:
def canal_inicial(evento):
    iniciais = [p.pdg_id for p in evento.particulas if p.status == -1]
    return " + ".join(f"PDG {pdg}" for pdg in sorted(iniciais))

def topologia_esperada(evento):
    finais = Counter(p.pdg_id for p in evento.particulas if p.status == 1)
    return finais[22] == 2

linhas_qualidade = []
linhas_canais = []
for amostra, lista_eventos in eventos.items():
    nup = np.array([len(evento.particulas) for evento in lista_eventos])
    pesos = np.array([evento.peso for evento in lista_eventos], dtype=float)
    campos_finitos = [
        all(np.isfinite([p.px, p.py, p.pz, p.energia, p.massa]).all() for p in evento.particulas)
        for evento in lista_eventos
    ]
    linhas_qualidade.append({
        "Amostra": amostra, "Eventos": len(lista_eventos),
        "NUP mínimo": nup.min(), "NUP máximo": nup.max(),
        "NUP mais frequente": Counter(nup).most_common(1)[0][0],
        "Eventos com campos inválidos": len(campos_finitos) - sum(campos_finitos),
        "Topologia inesperada": sum(not topologia_esperada(e) for e in lista_eventos),
    })
    for canal, quantidade in Counter(canal_inicial(e) for e in lista_eventos).most_common():
        linhas_canais.append({
            "Amostra": amostra, "Canal inicial": canal,
            "Eventos": quantidade, "Fração (%)": 100 * quantidade / len(lista_eventos),
        })

qualidade_amostras = pd.DataFrame(linhas_qualidade)
canais_iniciais = pd.DataFrame(linhas_canais)
display(qualidade_amostras)
display(canais_iniciais.style.format({"Fração (%)": "{:.2f}"}))

## 2b. Investigação preliminar dos eventos

No padrão LHE, os `status` usados aqui têm a seguinte interpretação:

- `status = -1`: partícula inicial;
- `status = 1`: partícula final estável no nível gerado;
- `status = 2`: partícula intermediária cuja história foi preservada.

A tabela conta tanto o número total de ocorrências quanto quantos eventos contêm cada combinação de partícula e `status`.

In [ ]:
NOMES_PDG = {
    -25: "anti-H", 25: "H", 23: "Z", 24: "W+", -24: "W-",
    21: "g", 5: "b", -5: "b̄", 4: "c", -4: "c̄",
    3: "s", -3: "s̄", 2: "u", -2: "ū", 1: "d", -1: "d̄",
    13: "μ−", -13: "μ+", 11: "e−", -11: "e+",
    12: "νe", -12: "ν̄e", 14: "νμ", -14: "ν̄μ",
    16: "ντ", -16: "ν̄τ",
}
DESCRICAO_STATUS = {-1: "inicial", 1: "final", 2: "intermediária"}

def nome_particula(pdg_id):
    return NOMES_PDG.get(pdg_id, f"PDG {pdg_id}")

def tabela_particulas(amostra, lista_eventos):
    ocorrencias = Counter()
    eventos_com_particula = Counter()

    for evento in lista_eventos:
        chaves_do_evento = set()
        for particula in evento.particulas:
            chave = (particula.pdg_id, particula.status)
            ocorrencias[chave] += 1
            chaves_do_evento.add(chave)
        eventos_com_particula.update(chaves_do_evento)

    linhas = []
    for (pdg_id, status), quantidade in ocorrencias.items():
        n_eventos = eventos_com_particula[(pdg_id, status)]
        linhas.append({
            "Amostra": amostra, "PDG ID": pdg_id,
            "Partícula": nome_particula(pdg_id), "Status": status,
            "Papel": DESCRICAO_STATUS.get(status, "outro"),
            "Ocorrências": quantidade, "Eventos": n_eventos,
            "Fração dos eventos (%)": 100 * n_eventos / len(lista_eventos),
        })
    return pd.DataFrame(linhas).sort_values(["Status", "PDG ID"]).reset_index(drop=True)

tabelas = [tabela_particulas(amostra, lista) for amostra, lista in eventos.items()]
tabela_todas_particulas = pd.concat(tabelas, ignore_index=True)
display(tabela_todas_particulas.style.format({"Fração dos eventos (%)": "{:.2f}"}))

### Identificação física das amostras

Na amostra de **sinal**, o bóson de Higgs (`PDG 25`) decai em dois fótons (`PDG 22`). Na amostra de **fundo**, o mesmo estado final `γγ` é produzido pelo contínuo, sem exigir um Higgs intermediário. Como sinal e fundo têm o mesmo estado final observável, a separação deve explorar a massa invariante e as distribuições cinemáticas dos fótons.

## 3. Exploração do conteúdo

Selecionaremos partículas com `status = 1` e removeremos neutrinos (`|PDG| = 12, 14, 16`). Para cada partícula visível, calcularemos:

$$p_T = \sqrt{p_x^2+p_y^2}, \qquad \eta = \operatorname{asinh}(p_z/p_T), \qquad \phi = \operatorname{atan2}(p_y,p_x).$$

Os histogramas usarão exatamente os intervalos solicitados: $0<p_T<100$ GeV, $-3<\eta<3$ e $-\pi<\phi<\pi$.

In [ ]:
PDGS_NEUTRINOS = {12, 14, 16}

def particulas_finais_visiveis(evento):
    return [
        p for p in evento.particulas
        if p.status == 1 and abs(p.pdg_id) not in PDGS_NEUTRINOS
    ]

def criar_tabela_cinematica(eventos_por_amostra):
    linhas = []
    for amostra, lista_eventos in eventos_por_amostra.items():
        for indice_evento, evento in enumerate(lista_eventos, start=1):
            for particula in particulas_finais_visiveis(evento):
                linhas.append({
                    "Amostra": amostra, "Evento": indice_evento,
                    "PDG ID": particula.pdg_id,
                    "Partícula": nome_particula(particula.pdg_id),
                    "pT [GeV]": particula.pt,
                    "eta": particula.eta, "phi": particula.phi,
                })
    return pd.DataFrame(linhas)

dados_cinematicos = criar_tabela_cinematica(eventos)
print(f"Total de partículas finais visíveis: {len(dados_cinematicos):,}")
display(dados_cinematicos.head())

In [ ]:
resumo_cinematico = (
    dados_cinematicos
    .groupby(["Amostra", "PDG ID", "Partícula"])[["pT [GeV]", "eta", "phi"]]
    .agg(["count", "mean", "std", "min", "median", "max"])
)
display(resumo_cinematico.round(3))

In [ ]:
CONFIG_VARIAVEIS = {
    "pT [GeV]": {"intervalo": (0, 100), "rotulo": r"$p_T$ [GeV]"},
    "eta": {"intervalo": (-3, 3), "rotulo": r"$\eta$"},
    "phi": {"intervalo": (-math.pi, math.pi), "rotulo": r"$\phi$ [rad]"},
}
CORES = {"Sinal": "tab:blue", "Fundo": "tab:orange"}

def nome_seguro(pdg_id):
    return f"pdg_{'menos_' if pdg_id < 0 else ''}{abs(pdg_id)}"

def plotar_comparacao(dados, titulo_extra="", sufixo_arquivo="antes_cortes"):
    registros = []
    pdgs = sorted(dados["PDG ID"].unique(), key=lambda x: (abs(x), x))

    for pdg_id in pdgs:
        fig, eixos = plt.subplots(1, 3, figsize=(16, 4.2))
        nome = nome_particula(int(pdg_id))

        for eixo, (variavel, config) in zip(eixos, CONFIG_VARIAVEIS.items()):
            minimo, maximo = config["intervalo"]
            for amostra in ("Sinal", "Fundo"):
                selecao = dados[(dados["Amostra"] == amostra) & (dados["PDG ID"] == pdg_id)]
                no_intervalo = selecao[selecao[variavel].between(minimo, maximo, inclusive="both")]
                n_eventos = no_intervalo["Evento"].nunique()
                contagens, bordas = np.histogram(no_intervalo[variavel], bins=40, range=(minimo, maximo))
                centros = (bordas[:-1] + bordas[1:]) / 2
                eixo.stairs(
                    contagens, bordas, linewidth=1.8, color=CORES[amostra],
                    label=f"{amostra}: {len(no_intervalo):,} entradas / {n_eventos:,} eventos",
                )
                eixo.errorbar(
                    centros, contagens, yerr=np.sqrt(contagens), fmt="none",
                    ecolor=CORES[amostra], elinewidth=0.8, alpha=0.65, capsize=1.5,
                )
                registros.append({
                    "Partícula": nome, "PDG ID": int(pdg_id),
                    "Variável": variavel, "Amostra": amostra,
                    "Entradas totais": len(selecao),
                    "Underflow": int((selecao[variavel] < minimo).sum()),
                    "Entradas no intervalo": len(no_intervalo),
                    "Overflow": int((selecao[variavel] > maximo).sum()),
                    "Eventos representados": n_eventos,
                    "Eventos da amostra": len(eventos[amostra]),
                })
            eixo.set_xlabel(config["rotulo"])
            eixo.set_ylabel("Entradas")
            eixo.legend(fontsize=8)

        fig.suptitle(f"{nome} (PDG {pdg_id}) — sinal × fundo{titulo_extra}")
        fig.tight_layout()
        destino = PASTA_GRAFICOS / f"{nome_seguro(int(pdg_id))}_{sufixo_arquivo}.png"
        fig.savefig(destino, dpi=150, bbox_inches="tight")
        plt.show()

    return pd.DataFrame(registros)

contagens_antes = plotar_comparacao(dados_cinematicos)

In [ ]:
contagens_antes["Cobertura dos eventos (%)"] = (
    100 * contagens_antes["Eventos representados"] / contagens_antes["Eventos da amostra"]
)
display(contagens_antes.style.format({"Cobertura dos eventos (%)": "{:.2f}"}))

### Os histogramas representam todos os eventos?

Não necessariamente. Há quatro motivos possíveis:

1. uma espécie de partícula pode não aparecer em todos os eventos;
2. um evento pode conter mais de uma partícula da mesma espécie, produzindo mais entradas do que eventos;
3. neutrinos e partículas com `status != 1` foram removidos por definição;
4. uma partícula pode existir no evento, mas ficar fora do intervalo fixado para o histograma — especialmente $p_T>100$ GeV ou $|\eta|>3$.

Por isso, a tabela acima apresenta separadamente o número de **entradas** e o número de **eventos representados** em cada intervalo.

In [ ]:
cobertura_incompleta = contagens_antes[contagens_antes["Eventos representados"] < contagens_antes["Eventos da amostra"]]
if cobertura_incompleta.empty:
    print("Todos os eventos estão representados em todos os histogramas.")
else:
    print("Casos em que o intervalo do histograma não representa todos os eventos:")
    display(cobertura_incompleta[[
        "Partícula", "Variável", "Amostra",
        "Eventos representados", "Eventos da amostra",
    ]])

### Variáveis em nível de evento

Uma análise do LHC também organiza os objetos dentro de cada evento. A seguir distinguimos os fótons líder e sublíder em $p_T$ e calculamos $\Delta\eta$, $\Delta\phi$ e $\Delta R=\sqrt{(\Delta\eta)^2+(\Delta\phi)^2}$ para o par de fótons.

In [ ]:
def delta_phi(phi_1, phi_2):
    return (phi_1 - phi_2 + math.pi) % (2 * math.pi) - math.pi

def variaveis_do_par(particulas, rotulo):
    colunas = {
        f"pT {rotulo} líder [GeV]": np.nan,
        f"pT {rotulo} sublíder [GeV]": np.nan,
        f"Maior |eta| {rotulo}": np.nan,
        f"Delta eta {rotulo}{rotulo}": np.nan,
        f"Delta phi {rotulo}{rotulo}": np.nan,
        f"Delta R {rotulo}{rotulo}": np.nan,
    }
    if len(particulas) != 2:
        return colunas
    lider, sublider = sorted(particulas, key=lambda p: p.pt, reverse=True)
    d_eta = lider.eta - sublider.eta
    d_phi = delta_phi(lider.phi, sublider.phi)
    colunas.update({
        f"pT {rotulo} líder [GeV]": lider.pt,
        f"pT {rotulo} sublíder [GeV]": sublider.pt,
        f"Maior |eta| {rotulo}": max(abs(lider.eta), abs(sublider.eta)),
        f"Delta eta {rotulo}{rotulo}": abs(d_eta),
        f"Delta phi {rotulo}{rotulo}": abs(d_phi),
        f"Delta R {rotulo}{rotulo}": math.hypot(d_eta, d_phi),
    })
    return colunas

def criar_tabela_eventos(eventos_por_amostra):
    linhas = []
    for amostra, lista_eventos in eventos_por_amostra.items():
        for indice_evento, evento in enumerate(lista_eventos, start=1):
            finais = particulas_finais_visiveis(evento)
            fotons = [p for p in finais if abs(p.pdg_id) == 22]
            linha = {
                "Amostra": amostra, "Evento": indice_evento,
                "Multiplicidade visível": len(finais),
                "Número de fótons": len(fotons),
                "Topologia esperada": len(fotons) == 2,
            }
            linha.update(variaveis_do_par(fotons, "γ"))
            linhas.append(linha)
    return pd.DataFrame(linhas)

dados_eventos = criar_tabela_eventos(eventos)
display(dados_eventos.head())
display(dados_eventos.groupby("Amostra").median(numeric_only=True).round(3))

In [ ]:
VARIAVEIS_EVENTO = {
    "pT γ líder [GeV]": (0, 150),
    "pT γ sublíder [GeV]": (0, 150),
    "Maior |eta| γ": (0, 3),
    "Delta eta γγ": (0, 6),
    "Delta phi γγ": (0, math.pi),
    "Delta R γγ": (0, 6),
}
fig, eixos = plt.subplots(2, 3, figsize=(16, 8))
for eixo, (variavel, intervalo) in zip(eixos.flat, VARIAVEIS_EVENTO.items()):
    for amostra in ("Sinal", "Fundo"):
        valores = dados_eventos.loc[dados_eventos["Amostra"] == amostra, variavel].dropna()
        contagens, bordas = np.histogram(valores, bins=40, range=intervalo)
        centros = (bordas[:-1] + bordas[1:]) / 2
        eixo.stairs(contagens, bordas, color=CORES[amostra], linewidth=1.7, label=amostra)
        eixo.errorbar(centros, contagens, yerr=np.sqrt(contagens), fmt="none", ecolor=CORES[amostra], alpha=0.6)
    eixo.set_xlabel(variavel)
    eixo.set_ylabel("Eventos")
    eixo.legend()
fig.suptitle("Variáveis em nível de evento — sinal × fundo")
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "variaveis_evento_antes_cortes.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Seleção cinemática e cutflow

Aplicaremos cortes aos fótons finais. Neste exemplo didático, exigimos $p_T>25$ GeV para o fóton sublíder, $|\eta|<2{,}5$ para ambos e $\Delta R>0{,}4$ entre eles. Esses valores são ilustrativos e não substituem uma seleção completa do CMS ou ATLAS.

A tabela de cutflow mostrará as eficiências incrementais e acumuladas. A razão $\epsilon_S/\epsilon_B$ mede apenas o ganho relativo da seleção; ela **não é uma significância estatística**.

In [ ]:
PT_GAMMA = 25.0
ETA_GAMMA = 2.5
DELTA_R_MINIMO = 0.4

ETAPAS = [
    ("Sem cortes", lambda df: pd.Series(True, index=df.index)),
    ("Topologia: 2 fótons", lambda df: df["Topologia esperada"]),
    ("Aceitação dos fótons", lambda df: (df["pT γ sublíder [GeV]"] > PT_GAMMA) & (df["Maior |eta| γ"] < ETA_GAMMA)),
    ("Separação angular do par γγ", lambda df: df["Delta R γγ"] > DELTA_R_MINIMO),
]

mascaras_acumuladas = {
    amostra: pd.Series(True, index=dados_eventos.index[dados_eventos["Amostra"] == amostra])
    for amostra in ("Sinal", "Fundo")
}
anteriores = {amostra: int(mascara.sum()) for amostra, mascara in mascaras_acumuladas.items()}
iniciais = anteriores.copy()
linhas_cutflow = []

for ordem, (nome_etapa, construir_mascara) in enumerate(ETAPAS):
    linha = {"Ordem": ordem, "Etapa": nome_etapa}
    for amostra in ("Sinal", "Fundo"):
        dados_amostra = dados_eventos[dados_eventos["Amostra"] == amostra]
        mascaras_acumuladas[amostra] &= construir_mascara(dados_amostra)
        restantes = int(mascaras_acumuladas[amostra].sum())
        linha[f"Eventos {amostra.lower()}"] = restantes
        linha[f"Eficiência incremental {amostra.lower()} (%)"] = 100 * restantes / anteriores[amostra] if anteriores[amostra] else 0
        linha[f"Eficiência acumulada {amostra.lower()} (%)"] = 100 * restantes / iniciais[amostra]
        anteriores[amostra] = restantes
    ef_s = linha["Eficiência acumulada sinal (%)"]
    ef_b = linha["Eficiência acumulada fundo (%)"]
    linha["Ganho relativo εS/εB"] = ef_s / ef_b if ef_b else np.inf
    linhas_cutflow.append(linha)

cutflow = pd.DataFrame(linhas_cutflow).drop(columns="Ordem")
display(cutflow.style.format({coluna: "{:.2f}" for coluna in cutflow.columns if "Eficiência" in coluna or "Ganho" in coluna}))

eventos_aprovados = pd.concat([
    dados_eventos[dados_eventos["Amostra"] == amostra].loc[mascaras_acumuladas[amostra], ["Amostra", "Evento"]]
    for amostra in ("Sinal", "Fundo")
], ignore_index=True)

In [ ]:
dados_apos_corte = dados_cinematicos.merge(
    eventos_aprovados, on=["Amostra", "Evento"], how="inner"
)
contagens_depois = plotar_comparacao(
    dados_apos_corte, titulo_extra=" após o cutflow completo",
    sufixo_arquivo="apos_cutflow",
)
display(contagens_depois)

## 5. Conclusões da Parte 1

- As duas amostras possuem o mesmo estado final em nível de gerador: dois fótons.
- A amostra de sinal contém um bóson de Higgs que decai em dois fótons; o fundo produz o mesmo estado final sem exigir um Higgs intermediário.
- A massa invariante do par de fótons é a variável central para visualizar a assinatura do Higgs.
- O cutflow permite identificar qual requisito rejeita mais fundo e qual é o custo acumulado em eficiência do sinal.
- Os resultados são de nível de gerador. Uma análise experimental exigiria reconstrução, eficiência, resolução, trigger e demais efeitos do detector.
- A normalização por seção de choque e luminosidade, assim como $S/B$ e $S/\sqrt{B}$, pertence à Parte 2.

In [ ]:
ultima_etapa = cutflow.iloc[-1]
ef_sinal = ultima_etapa["Eficiência acumulada sinal (%)"]
ef_fundo = ultima_etapa["Eficiência acumulada fundo (%)"]
ganho = ef_sinal / ef_fundo if ef_fundo else np.inf
rejeicoes = cutflow.iloc[1:].copy()
rejeicoes["Rejeição incremental do fundo (%)"] = 100 - rejeicoes["Eficiência incremental fundo (%)"]
etapa_mais_forte = rejeicoes.loc[rejeicoes["Rejeição incremental do fundo (%)"].idxmax()]
display(Markdown(
    f"**Resultado final do cutflow:** eficiência acumulada de {ef_sinal:.1f}% para o sinal "
    f"e {ef_fundo:.1f}% para o fundo. O ganho relativo $\\epsilon_S/\\epsilon_B$ é {ganho:.1f}. "
    f"A etapa que mais rejeitou fundo incrementalmente foi **{etapa_mais_forte['Etapa']}**. "
    "Esse ganho não deve ser interpretado como significância estatística."
))
print(f"Gráficos salvos em: {PASTA_GRAFICOS}")

## Respostas de referência verificadas nas amostras

Estas amostras representam `pp → H → γγ` (sinal) e o contínuo `pp → γγ` (fundo). A tabela de eventos deve mostrar dois fótons finais por evento, sem `NaN` nas variáveis do par. O sinal deve apresentar um pico próximo de 125 GeV na massa invariante $m_{γγ}$, enquanto o fundo forma uma distribuição contínua.

Os números exatos de eventos, eficiências, yields e significâncias devem ser obtidos executando as células; eles dependem dos cortes e da luminosidade definidos no notebook. A interpretação deve deixar claro que os eventos estão em nível de gerador e não incluem resolução ou reconstrução real do detector.

# Parte 2 — conservação, massas e normalização

A seguir, a análise continua com as grandezas físicas e a normalização da Parte 2, mantendo a abordagem baseada em DataFrame.


In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import gzip
import math
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_rows", 120)

def localizar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    for candidato in candidatos:
        if (candidato / "data" / "sinal.lhe.gz").exists():
            return candidato.resolve()
    raise FileNotFoundError(
        "Nao encontrei data/sinal.lhe.gz. Abra o notebook a partir da raiz "
        "do projeto ou da pasta notebooks/."
    )

RAIZ = localizar_raiz()
PASTA_DADOS = RAIZ / "data"
PASTA_GRAFICOS = RAIZ / "resultados" / "graficos"
PASTA_GRAFICOS.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "Sinal": PASTA_DADOS / "sinal.lhe.gz",
    "Fundo": PASTA_DADOS / "fundo.lhe.gz",
}

LUMINOSIDADE_PB = 10000.0
print(f"Raiz do projeto: {RAIZ}")


## 1. Leitura, metadados e selecao da Parte 1


In [ ]:
def abrir_lhe(caminho):
    caminho = Path(caminho)
    if caminho.suffix == ".gz":
        return gzip.open(caminho, "rt", encoding="utf-8", errors="replace")
    return caminho.open("rt", encoding="utf-8", errors="replace")

def extrair_processo_madgraph(caminho):
    trecho = []
    dentro_do_cartao = False
    with abrir_lhe(caminho) as arquivo:
        for linha in arquivo:
            if "<MG5ProcCard>" in linha:
                dentro_do_cartao = True
            elif "</MG5ProcCard>" in linha:
                break
            elif dentro_do_cartao:
                trecho.append(linha)
    comandos = re.findall(r"^\s*generate\s+(.+?)\s*$", "".join(trecho), re.MULTILINE)
    return comandos[-1] if comandos else "processo nao identificado"

def extrair_init(caminho):
    linhas = []
    dentro_init = False
    with abrir_lhe(caminho) as arquivo:
        for linha in arquivo:
            limpa = linha.strip()
            if limpa == "<init>":
                dentro_init = True
                continue
            if limpa == "</init>":
                break
            if dentro_init and limpa and not limpa.startswith("<"):
                linhas.append(limpa)
    feixes = linhas[0].split()
    processo = linhas[1].split()
    return {
        "ebeam1": float(feixes[2]),
        "ebeam2": float(feixes[3]),
        "IDWTUP": int(feixes[8]),
        "NPRUP": int(feixes[9]),
        "sigma_pb": float(processo[0]),
        "erro_sigma_pb": float(processo[1]),
        "peso_maximo": float(processo[2]),
        "LPRUP": int(processo[3]),
    }

def ler_lhe_para_dataframes(arquivos):
    eventos_linhas = []
    particulas_linhas = []
    for amostra, caminho in arquivos.items():
        dentro_evento = False
        esperando_cabecalho = False
        restantes = 0
        indice_evento = 0
        with abrir_lhe(caminho) as arquivo:
            for numero_linha, linha in enumerate(arquivo, start=1):
                linha = linha.strip()
                if linha == "<event>":
                    dentro_evento = True
                    esperando_cabecalho = True
                    indice_evento += 1
                    continue
                if not dentro_evento:
                    continue
                if esperando_cabecalho:
                    if not linha or linha.startswith("#"):
                        continue
                    campos = linha.split()
                    if len(campos) < 6:
                        raise ValueError(f"Cabecalho invalido na linha {numero_linha}")
                    eventos_linhas.append({
                        "Amostra": amostra, "Evento": indice_evento,
                        "NUP": int(campos[0]), "IDPRUP": int(campos[1]),
                        "peso_evento": float(campos[2]), "SCALUP": float(campos[3]),
                        "AQEDUP": float(campos[4]), "AQCDUP": float(campos[5]),
                    })
                    restantes = int(campos[0])
                    esperando_cabecalho = False
                    continue
                if restantes > 0:
                    campos = linha.split()
                    if len(campos) < 13:
                        raise ValueError(f"Particula invalida na linha {numero_linha}")
                    particulas_linhas.append({
                        "Amostra": amostra, "Evento": indice_evento,
                        "PDG ID": int(campos[0]), "status": int(campos[1]),
                        "mae_1": int(campos[2]), "mae_2": int(campos[3]),
                        "cor_1": int(campos[4]), "cor_2": int(campos[5]),
                        "px": float(campos[6]), "py": float(campos[7]),
                        "pz": float(campos[8]), "energia": float(campos[9]),
                        "massa": float(campos[10]), "tempo_vida": float(campos[11]),
                        "spin": float(campos[12]),
                    })
                    restantes -= 1
                    if restantes == 0:
                        dentro_evento = False
    eventos_df = pd.DataFrame(eventos_linhas)
    particulas_df = pd.DataFrame(particulas_linhas)
    particulas_df["pT [GeV]"] = np.hypot(particulas_df["px"], particulas_df["py"])
    particulas_df["eta"] = np.arcsinh(particulas_df["pz"] / particulas_df["pT [GeV]"].replace(0, np.nan))
    particulas_df["phi"] = np.arctan2(particulas_df["py"], particulas_df["px"])
    return eventos_df, particulas_df

@dataclass(frozen=True)
class Particula:
    pdg_id: int
    status: int
    mae_1: int
    mae_2: int
    cor_1: int
    cor_2: int
    px: float
    py: float
    pz: float
    energia: float
    massa: float
    tempo_vida: float
    spin: float

    @property
    def pt(self):
        return math.hypot(self.px, self.py)

    @property
    def eta(self):
        if self.pt == 0:
            return math.copysign(math.inf, self.pz)
        return math.asinh(self.pz / self.pt)

    @property
    def phi(self):
        return math.atan2(self.py, self.px)

@dataclass(frozen=True)
class Evento:
    processo_id: int
    peso: float
    escala: float
    alpha_qed: float
    alpha_qcd: float
    particulas: tuple

def converter_dataframes_para_eventos(eventos_df, particulas_df):
    eventos_por_amostra = {}
    for amostra, tabela_eventos in eventos_df.groupby("Amostra", sort=False):
        lista = []
        tabela_particulas_amostra = particulas_df[particulas_df["Amostra"] == amostra]
        for linha_evento in tabela_eventos.itertuples(index=False):
            partes = tabela_particulas_amostra[tabela_particulas_amostra["Evento"] == linha_evento.Evento]
            particulas = tuple(
                Particula(
                    pdg_id=int(linha["PDG ID"]), status=int(linha["status"]),
                    mae_1=int(linha["mae_1"]), mae_2=int(linha["mae_2"]),
                    cor_1=int(linha["cor_1"]), cor_2=int(linha["cor_2"]),
                    px=float(linha["px"]), py=float(linha["py"]), pz=float(linha["pz"]),
                    energia=float(linha["energia"]), massa=float(linha["massa"]),
                    tempo_vida=float(linha["tempo_vida"]), spin=float(linha["spin"]),
                )
                for _, linha in partes.iterrows()
            )
            dados = linha_evento._asdict()
            lista.append(Evento(
                processo_id=int(dados["IDPRUP"]), peso=float(dados["peso_evento"]),
                escala=float(dados["SCALUP"]), alpha_qed=float(dados["AQEDUP"]),
                alpha_qcd=float(dados["AQCDUP"]), particulas=particulas,
            ))
        eventos_por_amostra[amostra] = lista
    return eventos_por_amostra

metadados = {amostra: extrair_init(caminho) for amostra, caminho in ARQUIVOS.items()}
eventos_df, particulas_df = ler_lhe_para_dataframes(ARQUIVOS)
eventos = converter_dataframes_para_eventos(eventos_df, particulas_df)

display(eventos_df.head())
display(particulas_df.head())
for amostra, caminho in ARQUIVOS.items():
    print(f"{amostra}: {extrair_processo_madgraph(caminho)}")
    print(f"  eventos: {len(eventos[amostra]):,}")
    print(f"  sigma = {metadados[amostra]['sigma_pb']:.6g} pb")


In [ ]:
NOMES_PDG = {
    25: "H", 23: "Z", 22: "γ", 21: "g", 5: "b", -5: "bbar",
    4: "c", -4: "cbar", 3: "s", -3: "sbar",
    2: "u", -2: "ubar", 1: "d", -1: "dbar",
}

def nome_particula(pdg_id):
    return NOMES_PDG.get(pdg_id, f"PDG {pdg_id}")

def quadrimomento(particula):
    return np.array([particula.energia, particula.px, particula.py, particula.pz], dtype=float)

def soma_quadrimomentos(particulas):
    if not particulas:
        return np.zeros(4)
    return np.sum([quadrimomento(p) for p in particulas], axis=0)

def massa2(vetor):
    energia, px, py, pz = vetor
    return energia**2 - px**2 - py**2 - pz**2

def massa_invariante(particulas):
    return math.sqrt(max(massa2(soma_quadrimomentos(particulas)), 0.0))

def pt_sistema(particulas):
    vetor = soma_quadrimomentos(particulas)
    return math.hypot(vetor[1], vetor[2])

def eta_sistema(particulas):
    vetor = soma_quadrimomentos(particulas)
    pt = math.hypot(vetor[1], vetor[2])
    if pt == 0:
        return math.copysign(math.inf, vetor[3])
    return math.asinh(vetor[3] / pt)

def delta_phi(phi_1, phi_2):
    return (phi_1 - phi_2 + math.pi) % (2 * math.pi) - math.pi

def delta_r(p1, p2):
    return math.hypot(p1.eta - p2.eta, delta_phi(p1.phi, p2.phi))

def particulas_por_status(evento, status):
    return [p for p in evento.particulas if p.status == status]

def finais_visiveis(evento):
    return [p for p in evento.particulas if p.status == 1 and abs(p.pdg_id) not in {12, 14, 16}]

def par_fotons(evento):
    return [p for p in finais_visiveis(evento) if abs(p.pdg_id) == 22]

def evento_passa_parte1(evento):
    fotons = par_fotons(evento)
    if len(fotons) != 2:
        return False
    if min(p.pt for p in fotons) <= 25.0 or max(abs(p.eta) for p in fotons) >= 2.5:
        return False
    if delta_r(fotons[0], fotons[1]) <= 0.4:
        return False
    return True

linhas_cutflow = []
for amostra, lista_eventos in eventos.items():
    etapas = [
        ("Sem cortes", lista_eventos),
        ("Topologia esperada", [e for e in lista_eventos if len(par_fotons(e)) == 2]),
        ("Selecao final Parte 1", [e for e in lista_eventos if evento_passa_parte1(e)]),
    ]
    for etapa, selecionados in etapas:
        linhas_cutflow.append({
            "Amostra": amostra,
            "Etapa": etapa,
            "Eventos": len(selecionados),
            "Eficiencia (%)": 100 * len(selecionados) / len(lista_eventos),
        })

cutflow_resumido = pd.DataFrame(linhas_cutflow)
display(cutflow_resumido)

## 2. Conservacao do quadrimomento e `s_hat`


In [ ]:
indices = [1 + 1000 * i for i in range(10)]
linhas = []
for amostra, lista_eventos in eventos.items():
    for indice in indices:
        evento = lista_eventos[indice - 1]
        iniciais = particulas_por_status(evento, -1)
        finais = particulas_por_status(evento, 1)
        visiveis = finais_visiveis(evento)
        p_inicial = soma_quadrimomentos(iniciais)
        p_final = soma_quadrimomentos(finais)
        p_visivel = soma_quadrimomentos(visiveis)
        residuo = p_inicial - p_final
        linhas.append({
            "Amostra": amostra,
            "Evento": indice,
            "s_hat inicial [GeV^2]": massa2(p_inicial),
            "m2 finais [GeV^2]": massa2(p_final),
            "m2 visivel [GeV^2]": massa2(p_visivel),
            "max |Delta p_mu| [GeV]": np.max(np.abs(residuo)),
        })

consistencia = pd.DataFrame(linhas)
display(consistencia.round(6))
display(
    consistencia.groupby("Amostra")["max |Delta p_mu| [GeV]"]
    .max()
    .rename("Maior residuo nos eventos testados [GeV]")
    .reset_index()
)


## 3. Massas invariantes e histogramas 2D


In [ ]:
linhas_massas = []
for amostra, lista_eventos in eventos.items():
    for indice, evento in enumerate(lista_eventos, start=1):
        fotons = par_fotons(evento)
        visiveis = finais_visiveis(evento)
        if len(fotons) == 2:
            linhas_massas.append({
                "Amostra": amostra,
                "Evento": indice,
                "Passa selecao": evento_passa_parte1(evento),
                "m_gg [GeV]": massa_invariante(fotons),
                "m_visivel [GeV]": massa_invariante(visiveis),
                "pT_gg [GeV]": pt_sistema(fotons),
                "eta_gg": eta_sistema(fotons),
                "DeltaR_gg": delta_r(fotons[0], fotons[1]),
            })

massas = pd.DataFrame(linhas_massas)
display(massas.groupby("Amostra")[["m_gg [GeV]", "m_visivel [GeV]"]].describe().round(3))

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.2))
variaveis = {
    "m_gg [GeV]": (0, 200, r"$m_{γγ}$ [GeV]"),
    "m_visivel [GeV]": (0, 200, r"$m_{visível}$ [GeV]"),
}
for eixo, (variavel, (xmin, xmax, rotulo)) in zip(eixos, variaveis.items()):
    for amostra, cor in [("Fundo", "tab:orange"), ("Sinal", "tab:blue")]:
        valores = massas.loc[massas["Amostra"] == amostra, variavel]
        eixo.hist(valores, bins=50, range=(xmin, xmax), histtype="step", linewidth=1.8, label=amostra, color=cor)
    eixo.set_xlabel(rotulo)
    eixo.set_ylabel("Eventos")
    eixo.legend()
fig.suptitle("Massas invariantes do par de fótons")
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "parte2_massas_invariantes_formas.png", dpi=150, bbox_inches="tight")
plt.show()

fig, eixos = plt.subplots(1, 2, figsize=(11, 4.5))
for eixo, amostra in zip(eixos, ["Sinal", "Fundo"]):
    dados = massas[massas["Amostra"] == amostra]
    hist = eixo.hist2d(dados["m_gg [GeV]"], dados["pT_gg [GeV]"], bins=45, range=((0, 200), (0, 200)), cmap="viridis")
    eixo.set_title(amostra)
    eixo.set_xlabel(r"$m_{γγ}$ [GeV]")
    eixo.set_ylabel(r"$p_{T,γγ}$ [GeV]")
    fig.colorbar(hist[3], ax=eixo, label="Eventos")
fig.suptitle("Massa invariante γγ versus pT do sistema")
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "parte2_mgg_vs_ptgg.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Pesos, secao de choque e luminosidade


In [ ]:
linhas_norm = []
for amostra, lista_eventos in eventos.items():
    sigma = metadados[amostra]["sigma_pb"]
    n_gerado = len(lista_eventos)
    peso_fisico = sigma * LUMINOSIDADE_PB / n_gerado
    aprovados = [e for e in lista_eventos if evento_passa_parte1(e)]
    linhas_norm.append({
        "Amostra": amostra,
        "Eventos gerados": n_gerado,
        "sigma [pb]": sigma,
        "L [pb^-1]": LUMINOSIDADE_PB,
        "peso sigma*L/N": peso_fisico,
        "peso medio da simulacao": np.mean([evento.peso for evento in lista_eventos]),
        "yield total": sigma * LUMINOSIDADE_PB,
        "eventos aprovados": len(aprovados),
        "yield apos selecao": len(aprovados) * peso_fisico,
        "erro estatistico": math.sqrt(len(aprovados)) * peso_fisico,
    })

normalizacao = pd.DataFrame(linhas_norm)
display(normalizacao.round(6))

S = float(normalizacao.loc[normalizacao["Amostra"] == "Sinal", "yield apos selecao"].iloc[0])
B = float(normalizacao.loc[normalizacao["Amostra"] == "Fundo", "yield apos selecao"].iloc[0])
erro_S = float(normalizacao.loc[normalizacao["Amostra"] == "Sinal", "erro estatistico"].iloc[0])
erro_B = float(normalizacao.loc[normalizacao["Amostra"] == "Fundo", "erro estatistico"].iloc[0])
significancia_sb = S / math.sqrt(B) if B > 0 else np.nan
asimov = math.sqrt(2 * ((S + B) * math.log(1 + S / B) - S)) if B > 0 else np.nan
sigma_b_sist = 0.10 * B
significancia_sist = S / math.sqrt(B + sigma_b_sist**2) if B > 0 else np.nan

display(pd.DataFrame([{
    "S": S,
    "B": B,
    "S/B": S / B if B else np.nan,
    "S/sqrt(B)": significancia_sb,
    "Z Asimov": asimov,
    "S/sqrt(B + (0.10B)^2)": significancia_sist,
    "erro S": erro_S,
    "erro B": erro_B,
}]).round(6))


## 5. Histogramas empilhados normalizados


In [ ]:
selecionadas = massas[massas["Passa selecao"]].copy()
pesos_por_amostra = {
    linha["Amostra"]: linha["peso sigma*L/N"]
    for _, linha in normalizacao.iterrows()
}
selecionadas["peso"] = selecionadas["Amostra"].map(pesos_por_amostra)

def hist_ponderado(ax, variavel, intervalo, bins=50):
    fundo = selecionadas.loc[selecionadas["Amostra"] == "Fundo", variavel]
    sinal = selecionadas.loc[selecionadas["Amostra"] == "Sinal", variavel]
    wf = np.full(len(fundo), pesos_por_amostra["Fundo"])
    ws = np.full(len(sinal), pesos_por_amostra["Sinal"])
    cont_f, bordas = np.histogram(fundo, bins=bins, range=intervalo, weights=wf)
    cont_s, _ = np.histogram(sinal, bins=bins, range=intervalo, weights=ws)
    err2_f, _ = np.histogram(fundo, bins=bins, range=intervalo, weights=wf**2)
    err2_s, _ = np.histogram(sinal, bins=bins, range=intervalo, weights=ws**2)
    ax.stairs(cont_f, bordas, fill=True, alpha=0.45, color="tab:orange", label="Fundo")
    ax.stairs(cont_f + cont_s, bordas, fill=True, alpha=0.35, color="tab:blue", label="Sinal + fundo")
    centros = 0.5 * (bordas[:-1] + bordas[1:])
    ax.errorbar(centros, cont_f + cont_s, yerr=np.sqrt(err2_f + err2_s), fmt="none", color="black", linewidth=0.8, capsize=1.5)
    ax.set_ylabel("Eventos esperados")
    ax.legend()

fig, eixos = plt.subplots(1, 3, figsize=(16, 4.2))
for eixo, (variavel, (xmin, xmax, rotulo)) in zip(eixos, variaveis.items()):
    hist_ponderado(eixo, variavel, (xmin, xmax))
    eixo.set_xlabel(rotulo)
fig.suptitle(r"Expectativa MC normalizada para $L = 10/fb$ apos selecao")
fig.tight_layout()
fig.savefig(PASTA_GRAFICOS / "parte2_massas_normalizadas_empilhadas.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Estimativa simplificada da secao de choque


In [ ]:
linhas_sigma = []
for _, linha in normalizacao.iterrows():
    amostra = linha["Amostra"]
    n_gerado = linha["Eventos gerados"]
    n_pass = linha["eventos aprovados"]
    aceitacao_eficiencia = n_pass / n_gerado
    n_esperado = linha["yield apos selecao"]
    sigma_lhe = linha["sigma [pb]"]
    sigma_estimado = n_esperado / (aceitacao_eficiencia * LUMINOSIDADE_PB)
    linhas_sigma.append({
        "Amostra": amostra,
        "N_gerado": n_gerado,
        "N_pass": n_pass,
        "A*epsilon": aceitacao_eficiencia,
        "N esperado apos selecao": n_esperado,
        "sigma LHE [pb]": sigma_lhe,
        "sigma estimado [pb]": sigma_estimado,
        "diferenca relativa (%)": 100 * (sigma_estimado / sigma_lhe - 1),
    })

estimativa_sigma = pd.DataFrame(linhas_sigma)
display(estimativa_sigma.round(8))


## Respostas de referencia verificadas nas amostras

Estas amostras representam `sinal.lhe.gz` para `pp → H → γγ` e `fundo.lhe.gz` para o contínuo `pp → γγ`. A validação deve encontrar dois fótons finais por evento; por isso, as variáveis do par de fótons são preenchidas sem `NaN` quando a topologia esperada está presente.

A seleção da Parte 1 aplica cortes ilustrativos em `pT`, `|eta|` e `Delta R` dos fótons. Os números de eventos, eficiências, yields e significâncias devem ser obtidos executando as células, pois dependem dos cortes e da luminosidade definidos no notebook.

A massa invariante `m_gg` deve exibir o pico do Higgs próximo de 125 GeV no sinal, enquanto o fundo contínuo produz uma distribuição mais espalhada. A normalização e a estimativa simplificada de seção de choque são checagens didáticas baseadas nos metadados do LHE; não constituem uma medida experimental independente.

Os eventos permanecem em nível de gerador e não incluem reconstrução, resolução, identificação de fótons, trigger ou efeitos de pileup do detector.
